In [1]:
import mlflow
# Step 1: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://184.72.71.39:5000/")

c:\Users\Abhi\Documents\youtube-comment-analysis\mlops-youtube-comment-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location='s3://comment-analysis-bucket-994/6', creation_time=1788963985769, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1788963985769, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna

In [4]:
df = pd.read_csv('../data/processed/processed_comments.csv').dropna()
df.shape

(36662, 2)

In [5]:
# Remove rows with missing values
df = df.dropna(subset=['category', 'clean_comment'])

ngram_range = (1, 3)
max_features = 1000

# Final train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Separate validation data for Optuna
X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# TF-IDF fitted only on training data
vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_inner_vec = vectorizer.fit_transform(X_train_inner)
X_val_vec = vectorizer.transform(X_val)

# SMOTE only on training data
smote = SMOTE(random_state=42)

X_train_inner_vec, y_train_inner = smote.fit_resample(
    X_train_inner_vec,
    y_train_inner
)


# Optuna objective for SVM
def objective_svm(trial):

    C = trial.suggest_float(
        'C',
        1e-4,
        10.0,
        log=True
    )

    kernel = trial.suggest_categorical(
        'kernel',
        ['linear', 'rbf', 'poly']
    )

    model = SVC(
        C=C,
        kernel=kernel
    )

    model.fit(X_train_inner_vec, y_train_inner)

    y_pred = model.predict(X_val_vec)

    return accuracy_score(y_val, y_pred)


# Run Optuna
study = optuna.create_study(direction="maximize")

study.optimize(
    objective_svm,
    n_trials=20
)

best_params = study.best_params

print("Best parameters:", best_params)
print("Best validation accuracy:", study.best_value)


# Train final model using all training data
final_vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_vec = final_vectorizer.fit_transform(X_train)
X_test_vec = final_vectorizer.transform(X_test)

smote = SMOTE(random_state=42)

X_train_vec, y_train = smote.fit_resample(
    X_train_vec,
    y_train
)

best_model = SVC(
    C=best_params['C'],
    kernel=best_params['kernel']
)

best_model.fit(X_train_vec, y_train)

y_pred = best_model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)

print("Final accuracy:", accuracy)


# Log results in MLflow
with mlflow.start_run():

    mlflow.set_tag(
        "mlflow.runName",
        "SVM_SMOTE_TFIDF_Trigrams"
    )

    mlflow.set_tag(
        "experiment_type",
        "algorithm_comparison"
    )

    mlflow.log_param("algo_name", "SVM")
    mlflow.log_param("ngram_range", str(ngram_range))
    mlflow.log_param("max_features", max_features)
    mlflow.log_param("n_trials", 20)

    mlflow.log_params(best_params)

    mlflow.log_metric("accuracy", accuracy)

    classification_rep = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    for label, metrics in classification_rep.items():

        if isinstance(metrics, dict):

            for metric, value in metrics.items():

                mlflow.log_metric(
                    f"{label}_{metric}",
                    value
                )

    mlflow.sklearn.log_model(
        best_model,
        name="SVM_model"
    )

[I 2026-09-09 15:41:09,067] A new study created in memory with name: no-name-2b9726c4-d976-4229-9ca8-f1fa6a29d310
[I 2026-09-09 15:42:41,165] Trial 0 finished with value: 0.47954312990112513 and parameters: {'C': 0.0006781608503436121, 'kernel': 'linear'}. Best is trial 0 with value: 0.47954312990112513.
[I 2026-09-09 15:44:08,837] Trial 1 finished with value: 0.47954312990112513 and parameters: {'C': 0.000687366872684691, 'kernel': 'linear'}. Best is trial 0 with value: 0.47954312990112513.
[I 2026-09-09 15:45:38,138] Trial 2 finished with value: 0.47954312990112513 and parameters: {'C': 0.0017009118788404185, 'kernel': 'linear'}. Best is trial 0 with value: 0.47954312990112513.
[I 2026-09-09 15:50:08,902] Trial 3 finished with value: 0.5828503239004432 and parameters: {'C': 3.6097514431865165, 'kernel': 'poly'}. Best is trial 3 with value: 0.5828503239004432.
[I 2026-09-09 15:51:36,285] Trial 4 finished with value: 0.6004091374019775 and parameters: {'C': 0.006001231359882268, 'kerne

Best parameters: {'C': 0.7972219955526673, 'kernel': 'rbf'}
Best validation accuracy: 0.7921922945789295
Final accuracy: 0.7890358652666031
🏃 View run SVM_SMOTE_TFIDF_Trigrams at: http://184.72.71.39:5000/#/experiments/6/runs/c982b177528549d7a05d233c949b0037
🧪 View experiment at: http://184.72.71.39:5000/#/experiments/6


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['scipy.sparse._csr.csr_matrix'].

In [6]:
# Recreate the best SVM from the Optuna result
best_model = SVC(
    C=0.7972219955526673,
    kernel='rbf'
)

# Train the best model once
best_model.fit(X_train_vec, y_train)

# Final prediction
y_pred = best_model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)

print("Final accuracy:", accuracy)

classification_rep = classification_report(
    y_test,
    y_pred,
    output_dict=True
)


# Log the successful final run
with mlflow.start_run():

    mlflow.set_tag(
        "mlflow.runName",
        "SVM_SMOTE_TFIDF_Trigrams"
    )

    mlflow.set_tag(
        "experiment_type",
        "algorithm_comparison"
    )

    mlflow.log_param("algo_name", "SVM")
    mlflow.log_param("ngram_range", str((1, 3)))
    mlflow.log_param("max_features", 1000)
    mlflow.log_param("n_trials", 20)

    mlflow.log_param("C", 0.7972219955526673)
    mlflow.log_param("kernel", "rbf")

    mlflow.log_metric("accuracy", accuracy)

    for label, metrics in classification_rep.items():
        if isinstance(metrics, dict):
            for metric, value in metrics.items():
                mlflow.log_metric(
                    f"{label}_{metric}",
                    value
                )

    # Trust the scipy sparse matrix used internally by SVC
    mlflow.sklearn.log_model(
        best_model,
        name="SVM_model",
        skops_trusted_types=[
            "scipy.sparse._csr.csr_matrix"
        ]
    )

Final accuracy: 0.7890358652666031
🏃 View run SVM_SMOTE_TFIDF_Trigrams at: http://184.72.71.39:5000/#/experiments/6/runs/65b7adbeeb874104b142278e1d6644aa
🧪 View experiment at: http://184.72.71.39:5000/#/experiments/6
